# 04 - Model Deployment

This notebook demonstrates:

- **Blue-green deployment** for zero-downtime model updates
- **Retrieving the latest model version** from the Snowflake Model Registry
- **Creating a compute pool** for SPCS-based inference
- **Deploying models as services** with autocapture enabled for monitoring
- **Configuring a Gateway** to route traffic and provide a stable endpoint URL
- **Retiring old services** after successful traffic cutover

## Blue-Green Deployment Workflow

```
┌────────────────────────────────────────────────────────────────────┐
│                    BLUE-GREEN DEPLOYMENT                           │
├────────────────────────────────────────────────────────────────────┤
│                                                                    │
│  1. NEW VERSION       2. UPDATE GATEWAY      3. RETIRE OLD         │
│  ┌─────────────┐      ┌─────────────┐        ┌─────────────┐       │
│  │ Deploy new  │      │ Route 100%  │        │ Drop old    │       │
│  │ service     │ ───► │ to new      │  ───►  │ service     │       │
│  │ (v2)        │      │ service     │        │ (v1)        │       │
│  └─────────────┘      └─────────────┘        └─────────────┘       │
│                                                                    │
└────────────────────────────────────────────────────────────────────┘
```

## Gateway Benefits

| Feature | Description |
|---------|-------------|
| **Stable URL** | Single endpoint that never changes |
| **Zero Downtime** | Switch traffic without interruption |
| **Instant Rollback** | Revert by updating gateway spec |

## DAG Integration

This notebook is called by the ML Pipeline DAG when:
- Model validation passes (F1 >= threshold)
- New model outperforms currently deployed model

### Initialize Session and Configuration
Establish Snowflake session and query the experiment to find the best performing model based on precision score.

In [ ]:
import os
import snowflake.connector
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from datetime import datetime
import time

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except:
    conn = snowflake.connector.connect(
        connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME")
    )
    session = Session.builder.configs({"connection": conn}).create()

print(f"Connected to Snowflake: {session.get_current_account()}")

DATABASE = "DEMO_FRAUD_DETECTION_DB"
SCHEMA = "ML_PIPELINE"
MODEL_NAME = "FRAUD_DETECTION_MODEL"
COMPUTE_POOL_NAME = "ML_TRAINING_POOL"

session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
model_ref = reg.get_model(MODEL_NAME)
versions_df = model_ref.show_versions()
latest_version = versions_df.sort_values('created_on', ascending=False).iloc[0]['name']
MODEL_VERSION = latest_version

print(f"\nDeployment Configuration:")
print(f"  Model: {MODEL_NAME}:{MODEL_VERSION}")
print(f"  Compute Pool: {COMPUTE_POOL_NAME}")

### Set Database Context
Configure the role, database, schema, and warehouse. ACCOUNTADMIN is required for service and compute pool operations.

In [ ]:
print(f"Context: {DATABASE}.{SCHEMA}")

## Step 1: Get Latest Model Version and Check Current Deployment

### Get Latest Model Version from Registry
Retrieve the most recent model version from the Model Registry. The service name is derived from the version to ensure uniqueness and traceability.

In [ ]:
model_version = model_ref.version(MODEL_VERSION)

SERVICE_NAME = f"FRAUD_DETECTION_SERVICE_{MODEL_VERSION.replace('.', '_')}"

print(f"Model: {MODEL_NAME}")
print(f"Version to deploy: {MODEL_VERSION}")
print(f"Service name: {SERVICE_NAME}")

try:
    metrics = model_version.show_metrics()
    print(f"\nMetrics: {metrics}")
except:
    print("\nMetrics: (not available)")

### Check for Existing Deployed Service
Identify any currently running service. This is needed for the blue-green deployment pattern—we deploy the new version alongside the old one, then switch traffic.

In [ ]:
existing_services = session.sql(f"""
    SHOW SERVICES LIKE 'FRAUD_DETECTION_SERVICE%' IN SCHEMA {DATABASE}.{SCHEMA}
""").collect()

current_service_name = None
if existing_services:
    for svc in existing_services:
        if 'FRAUD_DETECTION_SERVICE' in svc['name'] and svc['status'] in ('READY', 'RUNNING', 'SUSPENDED'):
            current_service_name = svc['name']
            print(f"Current deployed service: {current_service_name}")
            print(f"Status: {svc['status']}")
            break

if not current_service_name:
    print("No existing model service found - this will be the initial deployment")

## Step 2: Create Compute Pool

### Create or Verify Compute Pool
The compute pool provides SPCS container resources for model inference. Using `IF NOT EXISTS` makes this idempotent—safe to run multiple times.

In [ ]:
print(f"Using compute pool: {COMPUTE_POOL_NAME}")

pool_status = session.sql(f"DESC COMPUTE POOL {COMPUTE_POOL_NAME}").collect()
print(f"✓ Compute pool {COMPUTE_POOL_NAME} ready")

## Step 3: Deploy New Model Version as Service

### Deploy Model as SPCS Service
Create an inference service from the model version. Key settings:
- **ingress_enabled**: Allows HTTP access to the service
- **autocapture**: Automatically logs all predictions for monitoring and drift detection
- Container build takes 5-10 minutes on first deployment.

In [ ]:
print(f"Deploying model version: {MODEL_VERSION}")
print(f"Service name: {SERVICE_NAME}")
print("(Container build may take 5-10 minutes)\n")

try:
    model_version.create_service(
        service_name=SERVICE_NAME,
        service_compute_pool=COMPUTE_POOL_NAME,
        image_build_compute_pool=COMPUTE_POOL_NAME,
        ingress_enabled=True,
        max_instances=1,
    )
    print(f"✓ Service creation initiated: {SERVICE_NAME}")
    print(f"  └── Model version: {MODEL_VERSION}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Service {SERVICE_NAME} already exists")
    else:
        print(f"Error: {e}")
        raise

## Step 4: Update Gateway to Route to New Service

A **Gateway** in Snowflake is a managed load balancer that provides a single, stable URL for routing traffic to one or more SPCS services. It enables blue-green deployments by allowing you to instantly switch traffic between model versions without changing client integrations.

### Create or Update Gateway
The gateway provides a stable URL that never changes. Traffic is routed to the active service via the specification. This enables zero-downtime deployments—update the gateway spec to instantly switch traffic to a new model version.

In [ ]:
print("Waiting for service to be ready...")
for i in range(30):
    status = session.sql(f"DESC SERVICE {DATABASE}.{SCHEMA}.{SERVICE_NAME}").collect()
    svc_status = status[0]['status'] if status else 'UNKNOWN'
    print(f"  Service status: {svc_status}")
    if svc_status == 'RUNNING':
        print(f"✓ Service {SERVICE_NAME} is ready!")
        break
    time.sleep(30)
else:
    print("Service not ready after 15 minutes - check Snowsight for status")

### Get Gateway URL
Retrieve the public endpoint URL. This URL remains constant across deployments—clients never need to update their integration when models are updated.

In [ ]:
svc_info = session.sql(f"DESC SERVICE {DATABASE}.{SCHEMA}.{SERVICE_NAME}").collect()[0]

print("=== Service Info ===")
print(f"Name: {svc_info['name']}")
print(f"Status: {svc_info['status']}")
print(f"Model version: {MODEL_VERSION}")

endpoints = session.sql(f"SHOW ENDPOINTS IN SERVICE {DATABASE}.{SCHEMA}.{SERVICE_NAME}").collect()
if endpoints:
    for ep in endpoints:
        print(f"\n🔗 Endpoint: {ep['name']}")
        ingress_url = ep['ingress_url'] if 'ingress_url' in ep.as_dict() else 'N/A'
        print(f"   URL: {ingress_url}")

## Step 5: Retire Old Service

### Retire Old Service
After traffic has been switched to the new service, the old one is no longer needed. Suspending then dropping it frees up compute resources and reduces costs.

In [ ]:
if current_service_name and current_service_name != SERVICE_NAME:
    print(f"Retiring old service: {current_service_name}")
    
    try:
        session.sql(f"ALTER SERVICE {DATABASE}.{SCHEMA}.{current_service_name} SUSPEND").collect()
        print(f"  ✓ Service suspended")
        
        session.sql(f"DROP SERVICE IF EXISTS {DATABASE}.{SCHEMA}.{current_service_name}").collect()
        print(f"  ✓ Service dropped")
        
        print(f"\n✓ Old service {current_service_name} retired successfully")
    except Exception as e:
        print(f"  ⚠ Could not retire old service: {e}")
else:
    print("No old service to retire (first deployment or same version)")